In [1]:
import zipfile, os

zip_path = "archive (30).zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir("/content/data")

['chest_xray']

In [2]:
import os

print(os.listdir("/content/data"))
print(os.listdir("/content/data/chest_xray"))

['chest_xray']
['chest_xray', 'test', 'train', 'val', '__MACOSX']


In [3]:
import shutil, os
mac = "/content/data/__MACOSX"
if os.path.exists(mac):
    shutil.rmtree(mac)

In [4]:
print(os.listdir("/content/data"))
print(os.listdir("/content/data/chest_xray"))

['chest_xray']
['chest_xray', 'test', 'train', 'val', '__MACOSX']


In [5]:
import tensorflow as tf

train_dir = "/content/data/chest_xray/train"
val_dir   = "/content/data/chest_xray/val"
test_dir  = "/content/data/chest_xray/test"

img_size = (224,224)
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb"
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb"
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb"
)

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.


In [6]:
img_size = (224,224)
batch_size = 32

train_dir = "/content/data/chest_xray/train"
val_dir   = "/content/data/chest_xray/val"
test_dir  = "/content/data/chest_xray/test"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb"
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb"
)


Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.


In [7]:
# Normalize
normalization = tf.keras.layers.Rescaling(1./255)

In [8]:
# Transfer Learning
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, input_shape=(224,224,3), weights="imagenet"
)
base_model.trainable = False

model = tf.keras.Sequential([
    normalization,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history = model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 250s 1s/step - accuracy: 0.6969 - auc: 0.4996 - loss: 0.6056 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8312
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.7429 - auc: 0.4804 - loss: 0.5767 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8343
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 176s 1s/step - accuracy: 0.7429 - auc: 0.4957 - loss: 0.5746 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8319
Epoch 4/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 178s 1s/step - accuracy: 0.7429 - auc: 0.5012 - loss: 0.5740 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8348
Epoch 5/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.7429 - auc: 0.5016 - loss: 0.5733 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8391


In [10]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

# Rebuild datasets with label_mode
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb",
    label_mode="binary"
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size, color_mode="rgb",
    label_mode="binary"
)

# Class weights (handle imbalance)

labels = np.concatenate([y for x, y in train_ds], axis=0)
labels = labels.reshape(-1)  # make 1-D
labels = labels.astype(int)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(class_weight_dict)
# Unfreeze top layers
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight_dict
)

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
{0: np.float64(1.9448173005219984), 1: np.float64(0.6730322580645162)}
Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 239s 1s/step - accuracy: 0.5322 - auc: 0.5004 - loss: 0.7004 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.8502
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 255s 1s/step - accuracy: 0.5109 - auc: 0.5066 - loss: 0.6986 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7553
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 215s 1s/step - accuracy: 0.5286 - auc: 0.5189 - loss: 0.6953 - val_accuracy: 0.5000 - val_auc: 0.7500 - val_loss: 0.7132
Epoch 4/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 264s 1s/step - accuracy: 0.5178 - auc: 0.5064 - loss: 0.6967 - val_accuracy: 0.5000 - val_auc: 0.5625 - val_loss: 0.6983
Epoch 5/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.5157 - auc: 0.5279 - loss: 0.6941 - val_accuracy: 0.5000 - val_auc: 0.6875 - val_loss: 0.7016


In [11]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.


In [12]:
img_size = (224,224)
batch_size = 32
train_dir = "/content/data/chest_xray/train"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

# Data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
])

# Model
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, input_shape=(224,224,3), weights="imagenet"
)
base_model.trainable = False

model = tf.keras.Sequential([
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history = model.fit(train_ds, validation_data=val_ds, epochs=5)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.
Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.7338 - auc: 0.4940 - loss: 0.5946 - val_accuracy: 0.7613 - val_auc: 0.5000 - val_loss: 0.5511
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.7383 - auc: 0.4990 - loss: 0.5791 - val_accuracy: 0.7613 - val_auc: 0.5694 - val_loss: 0.5509
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.7383 - auc: 0.4937 - loss: 0.5800 - val_accuracy: 0.7613 - val_auc: 0.5567 - val_loss: 0.5509
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.7383 - auc: 0.5025 - loss: 0.5787 - val_accuracy: 0.7613 - val_auc: 0.4868 - val_loss: 0.5504
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.7383 - auc: 0.4999 - loss: 0.5789 - val_accuracy: 0.7613 - val_auc: 0.4950 - val_loss: 0.5509


In [13]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# class weights
labels = np.concatenate([y for x, y in train_ds], axis=0).reshape(-1).astype(int)
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# unfreeze last layers
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weight_dict
)

Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 249s 2s/step - accuracy: 0.5152 - auc: 0.4967 - loss: 0.7003 - val_accuracy: 0.7613 - val_auc: 0.5000 - val_loss: 0.5559
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 267s 2s/step - accuracy: 0.5310 - auc: 0.5037 - loss: 0.7002 - val_accuracy: 0.7613 - val_auc: 0.5000 - val_loss: 0.5781
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 237s 2s/step - accuracy: 0.5095 - auc: 0.4984 - loss: 0.6996 - val_accuracy: 0.7613 - val_auc: 0.5000 - val_loss: 0.6216
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 233s 2s/step - accuracy: 0.5135 - auc: 0.5186 - loss: 0.6959 - val_accuracy: 0.7613 - val_auc: 0.4938 - val_loss: 0.6299
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 275s 2s/step - accuracy: 0.5183 - auc: 0.4953 - loss: 0.6996 - val_accuracy: 0.7613 - val_auc: 0.5442 - val_loss: 0.6419


In [14]:
img_size = (224,224)
batch_size = 32
train_dir = "/content/data/chest_xray/train"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=42,
    image_size=img_size, batch_size=batch_size
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=img_size, batch_size=batch_size
)

# class weights
labels = np.concatenate([y for x, y in train_ds], axis=0).reshape(-1).astype(int)
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# augmentation
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# preprocess for EfficientNet
preprocess = tf.keras.applications.efficientnet.preprocess_input

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(224,224,3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = data_aug(inputs)
x = preprocess(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc"), "accuracy"]
)

history = model.fit(
    train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight_dict
)

# fine-tune last 50 layers
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc"), "accuracy"]
)

history = model.fit(
    train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight_dict
)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.
Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 223s 2s/step - accuracy: 0.6350 - auc: 0.7155 - loss: 0.6224 - val_accuracy: 0.6155 - val_auc: 0.8790 - val_loss: 0.6520
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.7661 - auc: 0.8427 - loss: 0.5270 - val_accuracy: 0.7191 - val_auc: 0.9280 - val_loss: 0.5652
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.8018 - auc: 0.8905 - loss: 0.4651 - val_accuracy: 0.7747 - val_auc: 0.9451 - val_loss: 0.5031
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.8191 - auc: 0.9115 - loss: 0.4255 - val_accuracy: 0.7996 - val_auc: 0.9526 - val_loss: 0.4573
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.8421 - auc: 0.9315 - loss: 0.3840 - val_accuracy: 0.8054 - val_auc: 0.9570 - val_loss: 0.4350
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 260s 2s/s

In [16]:

test_dir = "/content/data/chest_xray/test"
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, image_size=img_size, batch_size=batch_size
)

model.evaluate(test_ds)


model.save("/content/xray_model.h5")

Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.8686 - auc: 0.9411 - loss: 0.3188


In [18]:
model.save("xray_model.keras") 